# 02 — Univariate Per-Pathway Signal

**H1 test**: for each of the 80 GapMind pathways, is pathway completeness significantly different between isolates and MAGs, both (a) pooled across the cohort and (b) within GTDB family (Mantel-Haenszel pooled OR, controlling for phylogenetic confounding)?

Pooled analysis confounds phylogeny with cultivation status — Pseudomonadota are over-represented among isolates, and they also happen to have richer carbon catabolism. The family-stratified Mantel-Haenszel OR is the appropriate effect size for the H1 hypothesis (does cultivation predict pathway content *within* lineage?).

Heavy lifting in `src/univariate_signal.py`; figures in `src/make_nb02_figures.py`.

In [1]:
!python ../src/univariate_signal.py 2>&1 | grep -E '^(Cohort|Pathways|Balanced|[0-9]+/80|MH q)'

Cohort: 235,671 genomes (208,073 isolates, 27,598 MAGs)
Pathways: 80
Balanced family subset: 212,121 genomes across 183 families
80/80 pathways had valid Mantel-Haenszel tests
MH q<0.05: 71 of 80 pathways


## Result summary

**Phylogeny-controlled Mantel-Haenszel (q<0.05, 183 GTDB families with ≥5 of each label):**

| Category | Isolate-enriched | MAG-enriched | NS or missing | Total |
|---|---:|---:|---:|---:|
| Amino acid biosynthesis | **16** | 1 | 1 | 18 |
| Carbon utilization | **47** | 7 | 8 | 62 |

Pooled vs Mantel-Haenszel comparison reveals **strong phylogenetic confounding**: many pathways with apparent pooled OR ≈ 8 collapse to within-family MH OR ≈ 1.3–1.9. Most of the pooled signal reflects which phyla are over-represented in cultured collections; the *within-family* signal — the relevant biology for cultivability prediction — is more modest but still highly significant.

## Top 10 pathways by raw isolate-vs-MAG gap

```
metabolic_category pathway_name  iso_complete_rate  mag_complete_rate  pooled_or  pooled_q  mh_or     mh_q
            carbon    D-alanine              0.598              0.153        8.2         0  0.973    0.438
            carbon      mannose              0.602              0.171       7.32         0   1.37        0
            carbon     glycerol               0.72              0.295       6.17         0   1.32        0
            carbon     D-serine              0.495              0.071       12.8         0   1.27 1.78e-10
            carbon     L-malate              0.659              0.236       6.24         0   1.39        0
            carbon      citrate              0.538              0.128       7.95         0   1.51        0
            carbon  glucosamine              0.602              0.193       6.33         0    1.9        0
            carbon    gluconate              0.515              0.121       7.69         0   1.49        0
            carbon     mannitol              0.469              0.105       7.48         0   1.32        0
            carbon       ribose              0.497              0.158       5.29         0   1.92        0
```

Note: pooled ORs are inflated by phylogeny. The Mantel-Haenszel ORs (mh_or) — typically 1.3–1.9 for these top pathways — are the family-controlled effect sizes that the predictive model in NB03 will exploit.

## Forest plot of top-30 pathways by |log₂ MH-OR|

![](../figures/per_pathway_forest.png)

Carbon utilization pathways (orange) dominate the top of the list — consistent with the Phase A finding that MAGs lag isolates by 28pp on carbon vs 9pp on amino-acid biosynthesis.

## Pooled vs family-controlled effect side-by-side

![](../figures/pooled_vs_mh_or.png)

The pooled-vs-MH gap quantifies how much of the raw isolate-MAG signal is phylogenetic. For most carbon pathways the pooled bar is ~2× the MH bar — meaning roughly half the apparent enrichment vanishes once we control for family identity.

## Distribution of per-genome pathway-completeness fractions

![](../figures/aa_vs_carbon_summary.png)

Visual confirmation of the Phase A signal: MAG distributions are shifted left (less complete) on both axes, but the shift is dramatically larger for carbon utilization. The bimodal isolate distribution on carbon (~0.4 hump + ~0.7 mode) likely reflects a copiotroph-vs-oligotroph dichotomy that NB03 will try to learn from.

## Hand-off to NB03

**H1 supported**: 71/80 pathways (88.7%) show family-controlled differential completeness; 63 enriched in isolates, 8 enriched in MAGs.

NB03 (`03_predictive_model.ipynb`) will treat the 80-dimensional binary completeness vector as the feature set for a leave-one-family-out (LOFO) classifier, benchmarked against CheckM-only and taxonomy-only baselines.